**LOADING DATA**

In [ ]:
import pandas as pd


In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
df=pd.read_csv('/content/drive/My Drive/Language_dataset.csv', encoding='latin1')

In [ ]:
df.tail(10)

In [ ]:
df.isnull().sum()

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df.columns

In [ ]:
df.head(5)

**DATA CLEANING**

In [ ]:
#Lowercasing
# The KeyError 'text' occurred because the column name was actually 'ï»¿text' due to a Byte Order Mark (BOM).
# First, ensure the column name is correctly 'text'.
if 'ï»¿text' in df.columns:
    df.rename(columns={'ï»¿text': 'text'}, inplace=True)

df['text'] = df['text'].str.lower()

In [ ]:
#Removing Puntuation
import string
string.punctuation
df['text']=df['text'].str.translate(str.maketrans('','',string.punctuation))
df.head(10)


In [ ]:
#remove extra space
df['text']=df['text'].str.strip()
df.tail(10)

In [ ]:
#remove any numbers and special characters
df['text'] = df['text'].str.replace(r'[^a-zA-Z\s]', '', regex=True)

In [ ]:
#tokenization
df['tokens'] = df['text'].apply(lambda x: x.split())
df.head(10)

**FEATURE EXTRACTION**

In [ ]:
#feature extraction using TF-IDf and n-grams
from numpy import vectorize
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer= TfidfVectorizer(analyzer='char',ngram_range=(2,4))
x = vectorizer.fit_transform(df['text'])
y = df['language']
print(x.shape)

In [ ]:
#model training using naive bayes
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=52)
model=MultinomialNB()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
print(accuracy_score(y_test,y_pred))
#test the model multiple times for better performance
#scores = cross_val_score(model, x, y, cv=5)
#print(scores.mean())


In [ ]:
#model training using logistic regression
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=1000)
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
print(accuracy_score(y_test,y_pred))
#test the model multiple times for better performance
#scores = cross_val_score(model, x, y, cv=5)
#print(scores.mean())


Logistic regression model has a better performance than naive bayes with a cross validation score of 88% accuracy while Naive has a 84% accuracy


**DEPLOYMENT**
Using logistic regression model


In [ ]:
import pickle

# Save model
pickle.dump(model, open('model.pkl', 'wb'))

# Save vectorizer
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))

In [ ]:
#save the file
from google.colab import files
files.download('model.pkl')
files.download('vectorizer.pkl')